# Notebook 1 — Analisi Prestazioni K-Fold

Valutazione sistematica di **Baseline CLAP** vs **ResiDual CLAP** su più dataset,
usando K-Fold stratificato per una stima robusta dell'accuracy.

**Pipeline per ogni dataset:**
1. Carica il dataset
2. Per ogni fold: split train/test stratificato
3. Baseline: valuta CLAP standard sul test fold
4. Residual: fitta PCA sul train fold → valuta sul test fold
5. Aggrega risultati → plot comparativi

In [2]:
# ============================================================
# INSTALL DEPENDENCIES (solo prima volta su Kaggle)
# ============================================================
#!pip install scikit-optimize --quiet

In [3]:
# ============================================================
# IMPORTS
# ============================================================
import sys
import os
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

# Aggiungi la root del progetto al path (adatta se necessario)
sys.path.insert(0, '/kaggle/working')
sys.path.insert(0, '/kaggle/input/clap-project')  # Adatta al tuo dataset Kaggle

from models.residual_clap_utils import (
    KFoldEvaluator,
    get_dataset_registry,
    plot_kfold_results,
)

print("✅ Import completati")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

✅ Import completati
Device: cpu


In [4]:
# ============================================================
# CONFIGURAZIONE GLOBALE
# ============================================================

# --- Parametri K-Fold ---
N_SPLITS     = 5       # Numero di fold
PCA_SAMPLES  = 100     # Campioni per fitting PCA (per fold)
TEST_SIZE    = 150     # Campioni di test per fold (None = usa tutti)
CLAP_VERSION = '2023'  # '2022' o '2023'
RANDOM_STATE = 42

# --- Configurazione ResiDual (punto di partenza) ---
RESIDUAL_CONFIG = {
    'mode': 'attention',
    'n_components_ratio': 0.15,
    'whitening_strength': 1.0,
    'whitening_eps': 1e-6,
    'target_layers': [3],
}

# --- Dataset da valutare ---
# Specifica qui quali dataset vuoi testare e il loro root path
DATASETS_TO_EVAL = {
    'ESC50':      '/home/lorenzo/Documenti/GitHub/ResiDual-CLAP/data/ESC-50-master',
    'VocalSound': '/home/lorenzo/Documenti/GitHub/ResiDual-CLAP/data/VocalSound',
    'TinySOL':  '/home/lorenzo/Documenti/GitHub/ResiDual-CLAP/data/TinySOL',
    #'IRMAS':    '/kaggle/input/irmas',
}

# Prompt testuale per la classificazione
PROMPT = 'this is the sound of '

print("Configurazione caricata:")
print(f"  N_SPLITS:     {N_SPLITS}")
print(f"  PCA_SAMPLES:  {PCA_SAMPLES}")
print(f"  TEST_SIZE:    {TEST_SIZE}")
print(f"  CLAP_VERSION: {CLAP_VERSION}")
print(f"  Config:       {RESIDUAL_CONFIG}")

Configurazione caricata:
  N_SPLITS:     5
  PCA_SAMPLES:  100
  TEST_SIZE:    150
  CLAP_VERSION: 2023
  Config:       {'mode': 'attention', 'n_components_ratio': 0.15, 'whitening_strength': 1.0, 'whitening_eps': 1e-06, 'target_layers': [3]}


In [5]:
# ============================================================
# CARICAMENTO DATASET
# ============================================================

DATASET_REGISTRY = get_dataset_registry()
print(f"Dataset disponibili: {list(DATASET_REGISTRY.keys())}")

loaded_datasets = {}

for ds_name, ds_root in DATASETS_TO_EVAL.items():
    if ds_name not in DATASET_REGISTRY:
        print(f"⚠️  {ds_name} non trovato nel registry, skip")
        continue
    
    print(f"\nCaricamento {ds_name} da {ds_root}...")
    try:
        ds = DATASET_REGISTRY[ds_name](root=ds_root, download=False)
        loaded_datasets[ds_name] = ds
        print(f"  ✅ {ds_name}: {len(ds)} campioni, {len(ds.classes)} classi")
        print(f"     Classi: {ds.classes}")
    except Exception as e:
        print(f"  ❌ Errore caricamento {ds_name}: {e}")

print(f"\nDataset caricati: {list(loaded_datasets.keys())}")

Dataset disponibili: ['ESC50', 'TinySOL', 'VocalSound', 'IRMAS']

Caricamento ESC50 da /home/lorenzo/Documenti/GitHub/ResiDual-CLAP/data/ESC-50-master...
Scaricando https://github.com/karoldvl/ESC-50/archive/master.zip ...


ESC-50-master.zip: 59.1MB [00:09, 6.81MB/s]


KeyboardInterrupt: 

In [ ]:
# ============================================================
# ESECUZIONE K-FOLD SU TUTTI I DATASET
# ============================================================

all_results = {}

for ds_name, dataset in loaded_datasets.items():
    print(f"\n{'='*70}")
    print(f"DATASET: {ds_name}  ({len(dataset)} campioni, {len(dataset.classes)} classi)")
    print(f"{'='*70}")
    
    text_labels = [PROMPT + x for x in dataset.classes]
    
    evaluator = KFoldEvaluator(
        dataset=dataset,
        text_labels=text_labels,
        residual_config=RESIDUAL_CONFIG,
        n_splits=N_SPLITS,
        pca_samples=PCA_SAMPLES,
        test_size=TEST_SIZE,
        clap_version=CLAP_VERSION,
        use_cuda=torch.cuda.is_available(),
        random_state=RANDOM_STATE,
    )
    
    results = evaluator.run(verbose=True)
    all_results[ds_name] = results
    
    # Salva risultato parziale per sicurezza
    with open(f'/kaggle/working/kfold_{ds_name}.json', 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print(f"\n💾 Risultati salvati in kfold_{ds_name}.json")

In [ ]:
# ============================================================
# RIEPILOGO TABELLARE
# ============================================================

print("\n" + "="*75)
print(f"{'RIEPILOGO FINALE':^75}")
print("="*75)
print(f"{'Dataset':<15} {'Baseline':>10} {'±std':>6} {'Residual':>10} {'±std':>6} {'Delta':>8}")
print("-"*75)

for ds_name, res in all_results.items():
    b = res['baseline']
    r = res['residual']
    delta = res['delta_mean']
    delta_str = f"{delta:+.4f}"
    print(f"{ds_name:<15} {b['mean']:>10.4f} {b['std']:>6.4f} "
          f"{r['mean']:>10.4f} {r['std']:>6.4f} {delta_str:>8}")

print("="*75)

In [ ]:
# ============================================================
# PLOT COMPARATIVI
# ============================================================

if len(all_results) > 0:
    fig = plot_kfold_results(all_results, figsize=(14, 5))
    plt.suptitle('ResiDual CLAP vs Baseline CLAP — Analisi K-Fold', 
                 fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig('/kaggle/working/kfold_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("📊 Plot salvato in kfold_comparison.png")

In [ ]:
# ============================================================
# PLOT PER-FOLD DETTAGLIATO (per ogni dataset)
# ============================================================

for ds_name, res in all_results.items():
    b_folds = res['baseline']['fold_accs']
    r_folds = res['residual']['fold_accs']
    n_folds = len(b_folds)
    fold_ids = list(range(1, n_folds + 1))
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'{ds_name} — Dettaglio K-Fold', fontsize=12)
    
    # Accuracy per fold
    ax = axes[0]
    ax.plot(fold_ids, b_folds, 'o-', color='steelblue', label='Baseline', alpha=0.8)
    ax.plot(fold_ids, r_folds, 's-', color='coral',     label='Residual', alpha=0.8)
    ax.axhline(res['baseline']['mean'], color='steelblue', linestyle='--', alpha=0.4)
    ax.axhline(res['residual']['mean'], color='coral',     linestyle='--', alpha=0.4)
    ax.set_xlabel('Fold')
    ax.set_ylabel('Accuracy')
    ax.set_title('Accuracy per fold')
    ax.set_xticks(fold_ids)
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1)
    
    # Delta per fold
    ax2 = axes[1]
    deltas = [r - b for r, b in zip(r_folds, b_folds)]
    colors = ['green' if d > 0 else 'red' for d in deltas]
    ax2.bar(fold_ids, deltas, color=colors, alpha=0.7)
    ax2.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax2.set_xlabel('Fold')
    ax2.set_ylabel('Δ Accuracy (Residual − Baseline)')
    ax2.set_title('Miglioramento per fold')
    ax2.set_xticks(fold_ids)
    ax2.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/kfold_{ds_name}_detail.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ============================================================
# SALVATAGGIO RISULTATI COMPLETI
# ============================================================

with open('/kaggle/working/kfold_all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print("✅ Tutti i risultati salvati in kfold_all_results.json")
print("\nFile generati:")
for f in Path('/kaggle/working').glob('kfold_*'):
    print(f"  {f.name}")

In [ ]:
# ============================================================
# ANALISI STATISTICA (t-test appaiato per fold)
# ============================================================

from scipy import stats

print("\n" + "="*60)
print("ANALISI STATISTICA (t-test appaiato per fold)")
print("="*60)

for ds_name, res in all_results.items():
    b_folds = np.array(res['baseline']['fold_accs'])
    r_folds = np.array(res['residual']['fold_accs'])
    
    if len(b_folds) < 2:
        print(f"{ds_name}: troppo pochi fold per t-test")
        continue
    
    t_stat, p_val = stats.ttest_rel(r_folds, b_folds)
    significance = "**" if p_val < 0.01 else ("*" if p_val < 0.05 else "ns")
    
    print(f"\n{ds_name}:")
    print(f"  t-statistic: {t_stat:+.4f}")
    print(f"  p-value:     {p_val:.4f}  ({significance})")
    print(f"  Conclusione: {'Miglioramento significativo' if p_val < 0.05 and t_stat > 0 else ('Peggioramento significativo' if p_val < 0.05 else 'Nessuna differenza significativa')}")